In [6]:
!git clone https://github.com/aedenj/faithful-cuts.git

Cloning into 'faithful-cuts'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 8 (delta 0), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (8/8), 37.45 KiB | 628.00 KiB/s, done.


In [1]:
!pip install qwen-omni-utils -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 87.2 MB/s eta 0:00:00


In [2]:
!pip install outlines pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.7/114.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.8 MB/s eta 0:00:00


# Setup

In [3]:
import requests
import gc
import json
import re
from pydantic import BaseModel, Field, ConfigDict

import outlines
from outlines.inputs import Chat, Video

import torch
from transformers.video_utils import load_video
import numpy as np

In [4]:
def cleanup_gpu():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# Extract Scenes

# Generate Scene Descriptions

### Model Setup

In [ ]:
from transformers import (
    Qwen3VLMoeForConditionalGeneration,
    AutoProcessor
)

SCENE_DESC_QWEN_VL30B = "Qwen/Qwen3-VL-30B-A3B-Instruct"

qwen_vl30b_model = Qwen3VLMoeForConditionalGeneration.from_pretrained(
    SCENE_DESC_QWEN_VL30B, dtype="auto", device_map="auto"
)

qwen_vl30b_processor = AutoProcessor.from_pretrained(SCENE_DESC_QWEN_VL30B)

In [5]:
from transformers import (
    AutoModelForMultimodalLM,
    AutoProcessor,
    # BitsAndBytesConfig,
)

SCENE_DESC_QWEN_3_5_35B = "Qwen/Qwen3.5-35B-A3B"


# quant_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )


qwen_3_5_35B_processor = AutoProcessor.from_pretrained(
    SCENE_DESC_QWEN_3_5_35B
)


qwen_3_5_35B_model = AutoModelForMultimodalLM.from_pretrained(
    SCENE_DESC_QWEN_3_5_35B,
    device_map="auto",
    dtype="auto",
)

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.54k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/1026 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

In [7]:
video_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/bottle-detection.mp4"
video_path = "sample_video.mp4"

try:
    with requests.get(video_url, stream=True, timeout=30) as r:
        r.raise_for_status()
        with open(video_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print(f"Successfully downloaded video to {video_path}")
except Exception as e:
    print(f"Failed to download video: {e}")

Successfully downloaded video to sample_video.mp4


In [8]:
frames, metadata = load_video(
    video_path,
    sample_indices_fn=lambda metadata, **kw: np.linspace(
        0, metadata.total_num_frames - 1, 64
    ).round().astype(int),
)

vp = qwen_3_5_35B_processor.video_processor
vp.do_sample_frames = False
vp.video_metadata = [metadata]

print(f"frames: {frames.shape}")
print(f"fps: {metadata.fps}, total: {metadata.total_num_frames}, duration: {metadata.duration}")

frames: (64, 360, 640, 3)
fps: 29.833333333333332, total: 1189, duration: 39.85474860335196


In [9]:
class SceneDescription(BaseModel):
    model_config = ConfigDict(extra="forbid")

    summary: str = Field(..., description="A concise but detailed summary of the visual scene depicted in the image or video.")

In [10]:
prompt = [
    {
        "role": "system",
        "content": (
            "You are a precise video understanding system. "
            "Report only information directly supported by the provided video."
        ),
    },
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": Video(frames),
            },
            {
                "type": "text",
                "text": """
Analyze this scene.

Describe what occurs in the scene based only on visually
observable evidence.

Describe people, objects, actions, interactions, locations,
settings, and important changes over time.

Do not infer information that cannot be established from the
video. Do not guess the identities or names of people.
""",
            },
        ],
    }
]

In [11]:
scene_desc_model = outlines.from_transformers(qwen_3_5_35B_model, qwen_3_5_35B_processor)

In [12]:
formatted_prompt = qwen_3_5_35B_processor.tokenizer.apply_chat_template(
    prompt,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

In [ ]:
result = scene_desc_model(
    [
        formatted_prompt,
        Video(frames),
    ],
    output_type=SceneDescription,
    max_new_tokens=1200,
    temperature=0.2,
)

In [ ]:
scene = SceneDescription.model_validate_json(result)
print(scene)

summary='A person is standing in front of a white wall with three bottles of water on a table in front of them. The person picks up the first bottle and drinks from it, then places it back on the table. They repeat this process with the second and third bottles. After drinking from all three bottles, the person walks away from the table.'


In [ ]:
del scene_desc_model
del qwen_3_5_35B_processor
del qwen_3_5_35B_model

cleanup_gpu()

In [ ]:
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   9344 KiB |  68905 MiB | 743422 MiB | 743413 MiB |
|       from large pool |   8320 KiB |  68811 MiB | 708607 MiB | 708599 MiB |
|       from small pool |   1024 KiB |     95 MiB |  34815 MiB |  34814 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   9344 KiB |  68905 MiB | 743422 MiB | 743413 MiB |
|       from large pool |   8320 KiB |  68811 MiB | 708607 MiB |

# Extract Facts

In [ ]:
!pip install -q openai

import os
for f in ("video_context_examples_0417.csv", "utils_video2text.py"):
    assert os.path.exists(f), f"upload {f} into the session working directory"

import utils_video2text as fifa

print(f"few-shot items: {len(fifa.TIFA160_ICL_TRAIN_IDS)}")
print(f"tuple examples: {len(fifa._TUPLE_EXAMPLES)}, question examples: {len(fifa._QUESTION_EXAMPLES)}")


In [ ]:
from typing import Literal
from pydantic import BaseModel, Field, ConfigDict

CategoryBroad = Literal[
    "entity", "attribute", "relation", "action", "event", "global", "other"
]
CategoryDetailed = Literal[
    "", "ambiguity", "color", "count", "material", "part", "scale", "shape",
    "spatial", "state", "temporal", "text rendering", "texture", "type", "whole",
]


class Fact(BaseModel):
    model_config = ConfigDict(extra="forbid")

    id: int
    category_broad: CategoryBroad
    category_detailed: CategoryDetailed = ""
    args: list[str] = Field(default_factory=list)


class FactList(BaseModel):
    model_config = ConfigDict(extra="forbid")

    facts: list[Fact]


In [ ]:
LLM_MODEL = "zai-org/GLM-4.7-Flash"



In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

glm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
glm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL, dtype="auto", device_map="auto"
)


def completion(prompt: str, max_tokens: int = 2048) -> str:
    messages = [
        {"role": "system", "content": "You are a language assistant."},
        {"role": "user", "content": prompt},
    ]
    text = glm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = glm_tokenizer([text], return_tensors="pt").to(glm_model.device)
    out = glm_model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    return glm_tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )


assert "</think>" in glm_tokenizer.apply_chat_template(
    [{"role": "user", "content": "x"}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False,
), "thinking not suppressed; reasoning trace will corrupt line parsing"


In [ ]:
import re
import collections

_TUPLE_RE = re.compile(
    r"^(?P<broad>[A-Za-z]+)\s*(?:-\s*(?P<detailed>[A-Za-z ]*?))?\s*\((?P<args>.*)\)\s*$"
)


def _clean_lines(output_str):
    if "</think>" in output_str:
        output_str = output_str.rsplit("</think>", 1)[1]
    if "output:" in output_str:
        output_str = output_str.split("output:", 1)[1]
    return [
        ln.strip() for ln in output_str.strip().split("\n")
        if ln.strip() and not ln.strip().startswith("```")
    ]


def _split_id(line):
    if "|" not in line:
        return None, None
    head, rest = line.split("|", 1)
    head = head.strip().lstrip("#").strip()
    return (int(head), rest.strip()) if head.isdigit() else (None, None)


def parse_tuples(output_str):
    facts, dropped, seen = [], [], set()
    for line in _clean_lines(output_str):
        tid, payload = _split_id(line)
        if tid is None or tid in seen:
            dropped.append(line)
            continue
        seen.add(tid)
        m = _TUPLE_RE.match(payload)
        facts.append({
            "id": tid,
            "raw": line,
            "text": payload,
            "category_broad": m.group("broad").strip().lower() if m else None,
            "category_detailed": (m.group("detailed") or "").strip().lower() if m else "",
            "args": [a.strip() for a in m.group("args").split(",") if a.strip()] if m else [],
        })
    facts.sort(key=lambda r: r["id"])
    return facts, dropped


def parse_questions(output_str):
    questions, dropped = {}, []
    for line in _clean_lines(output_str):
        qid, payload = _split_id(line)
        if qid is None or qid in questions:
            dropped.append(line)
            continue
        questions[qid] = payload
    return questions, dropped


def parse_report(facts, dropped, questions=None):
    uncategorized = [f["id"] for f in facts if f["category_broad"] is None]
    print(f"parsed {len(facts)} tuples, dropped {len(dropped)} lines, "
          f"{len(uncategorized)} uncategorized")
    print("categories:", dict(collections.Counter(f["category_broad"] for f in facts)))
    if questions is not None:
        ids_t, ids_q = {f["id"] for f in facts}, set(questions)
        print(f"questions {len(ids_q)} vs tuples {len(ids_t)}; "
              f"tuples without questions: {sorted(ids_t - ids_q)}; "
              f"questions without tuples: {sorted(ids_q - ids_t)}")
    for ln in dropped:
        print("  DROPPED:", ln[:100])


In [ ]:
DESCRIPTION = scene.summary
QUERY = "Please generate a caption for the video."

tuple_prompt = fifa.make_prompt(
    examples=fifa._TUPLE_EXAMPLES,
    test_input=DESCRIPTION,
    test_query=QUERY,
    preamble=fifa._TUPLE_PREAMBLE,
    task="tuple",
)

assert "query:" in tuple_prompt
assert tuple_prompt.rstrip().endswith("output:")
print(f"{len(tuple_prompt)} chars, {len(glm_tokenizer(tuple_prompt).input_ids)} tokens\n")
print(tuple_prompt[:600], "\n...\n", tuple_prompt[-500:])


In [ ]:
raw_tuples = fifa.parse_with_input_name(completion(tuple_prompt)).strip()
print(raw_tuples)

facts_a, dropped_t = parse_tuples(raw_tuples)
parse_report(facts_a, dropped_t)


In [ ]:
question_prompt = fifa.make_prompt(
    examples=fifa._QUESTION_EXAMPLES,
    test_input="\n".join([DESCRIPTION, raw_tuples]),
    test_query=QUERY,
    preamble=fifa._QUESTION_PREAMBLE,
    task="question",
)

raw_questions = fifa.parse_with_input_name(completion(question_prompt)).strip()
print(raw_questions)

questions_a, dropped_q = parse_questions(raw_questions)
parse_report(facts_a, dropped_q, questions_a)

arm_a = [
    {**f, "question": questions_a[f["id"]]}
    for f in facts_a if f["id"] in questions_a
]


In [ ]:
# outlines re-applies the chat template internally and cannot pass enable_thinking,
# so flip the template default instead.
_orig = glm_tokenizer.chat_template
glm_tokenizer.chat_template = _orig.replace(
    "enable_thinking is defined and not enable_thinking",
    "not (enable_thinking is defined and enable_thinking)",
)
assert glm_tokenizer.chat_template != _orig, "template patch did not apply"

import outlines

glm_struct = outlines.from_transformers(glm_model, glm_tokenizer)

ARM_B_PREAMBLE = fifa._TUPLE_PREAMBLE.rsplit("output format:", 1)[0].strip()

result_b = glm_struct(
    f"{ARM_B_PREAMBLE}\n\nquery: {QUERY}\ninput: {DESCRIPTION}",
    output_type=FactList,
    max_new_tokens=2048,
    do_sample=False,
)
facts_b = FactList.model_validate_json(result_b).facts

print(f"Arm A: {len(facts_a)} facts, Arm B: {len(facts_b)} facts")
print("A:", dict(collections.Counter(f["category_broad"] for f in facts_a)))
print("B:", dict(collections.Counter(f.category_broad for f in facts_b)))


In [ ]:
def gold_for(item_id):
    sub = fifa._TIFA160_DF[fifa._TIFA160_DF.item_id == item_id]
    return (sub.text.tolist()[0], sub.input_query.tolist()[0],
            sub.tuple.tolist(), sub.question_natural_language.tolist())


def build_examples(task, exclude_ids=()):
    ids = [i for i in fifa.TIFA160_ICL_TRAIN_IDS if i not in exclude_ids]
    return fifa.get_tifa_examples(fifa._TIFA160_DF, ids, task=task)


for item_id in ("video_001", "video_002", "video_003"):
    text, query, gold_tuples, gold_questions = gold_for(item_id)
    p = fifa.make_prompt(
        examples=build_examples("tuple", exclude_ids={item_id}),
        test_input=text, test_query=query,
        preamble=fifa._TUPLE_PREAMBLE, task="tuple",
    )
    pred, dropped = parse_tuples(fifa.parse_with_input_name(completion(p, 1024)).strip())
    gold_cats = collections.Counter(
        t.split("(")[0].split("-")[0].strip().lower() for t in gold_tuples
    )
    print(f"\n{item_id}: gold={len(gold_tuples)} pred={len(pred)} dropped={len(dropped)}")
    print("  gold:", dict(gold_cats))
    print("  pred:", dict(collections.Counter(f["category_broad"] for f in pred)))
    for g, pr in zip(gold_tuples, [f["text"] for f in pred]):
        print(f"    gold: {g:55s} pred: {pr}")


## Dependency Graph (STSDG)

In [ ]:
def parse_dependencies(output_str, valid_ids=None):
    """Parse `id | parents` lines into {id: [parent_ids]}.

    Root nodes are the empty list. The reference encodes roots as `0` and skips
    them at scoring time (eval_video2text.py:75), so dropping 0 is equivalent.
    """
    deps, dropped = {}, []
    for line in _clean_lines(output_str):
        did, payload = _split_id(line)
        if did is None or did in deps:
            dropped.append(line)
            continue
        parents = []
        for tok in payload.split(","):
            tok = tok.strip()
            if not tok.isdigit():
                continue
            pid = int(tok)
            if pid == 0 or pid == did or pid in parents:
                continue
            if valid_ids is not None and pid not in valid_ids:
                continue
            parents.append(pid)
        deps[did] = parents
    return deps, dropped


def validate_dsg(facts, deps):
    ids = {f["id"] for f in facts}
    report = {
        "missing_entry": sorted(ids - set(deps)),
        "unknown_id": sorted(set(deps) - ids),
        "dangling_parent": sorted(
            {p for d, ps in deps.items() for p in ps if p not in ids}
        ),
        "roots": sorted(d for d, ps in deps.items() if not ps),
        "forward_ref": sorted(d for d, ps in deps.items() if any(p > d for p in ps)),
    }

    colour, cycles = {}, []

    def walk(node, path):
        if colour.get(node) == 2:
            return
        if colour.get(node) == 1:
            cycles.append(path[path.index(node):] + [node])
            return
        colour[node] = 1
        for parent in deps.get(node, []):
            walk(parent, path + [node])
        colour[node] = 2

    for node in deps:
        walk(node, [])
    report["cycles"] = cycles

    print(f"nodes={len(ids)} edges={sum(len(p) for p in deps.values())} "
          f"roots={len(report['roots'])}")
    for key in ("missing_entry", "unknown_id", "dangling_parent", "forward_ref", "cycles"):
        if report[key]:
            print(f"  {key}: {report[key]}")
    if not any(report[k] for k in ("unknown_id", "dangling_parent", "cycles")):
        print("  graph is a well-formed DAG over the extracted tuples")
    return report


In [ ]:
dependency_prompt = fifa.make_prompt(
    examples=fifa._DEPENDENCY_EXAMPLES,
    test_input="\n".join([DESCRIPTION, raw_tuples]),
    test_query=QUERY,
    preamble=fifa._DEPENDENCY_PREAMBLE,
    task="dependency",
)

raw_dependencies = fifa.parse_with_input_name(completion(dependency_prompt)).strip()
print(raw_dependencies)

deps_a, dropped_d = parse_dependencies(raw_dependencies, valid_ids={f["id"] for f in facts_a})
for ln in dropped_d:
    print("  DROPPED:", ln[:100])

dsg_report = validate_dsg(facts_a, deps_a)


In [ ]:
stsdg = [
    {**f, "question": questions_a.get(f["id"]), "parents": deps_a.get(f["id"], [])}
    for f in facts_a
]

id2text = {f["id"]: f["text"] for f in facts_a}
for rec in stsdg:
    parents = ", ".join(id2text.get(p, f"?{p}") for p in rec["parents"]) or "-"
    print(f"{rec['id']:>3}  {rec['text']}")
    print(f"     Q: {rec['question']}")
    print(f"     parents: {parents}")

complete = [r for r in stsdg if r["question"] is not None]
print(f"\n{len(complete)}/{len(stsdg)} facts have a tuple, question and dependency entry")


In [ ]:
def gold_dependencies(item_id):
    sub = fifa._TIFA160_DF[fifa._TIFA160_DF.item_id == item_id]
    gold = {}
    for n, raw in enumerate(sub.dependency.tolist()):
        parents = [
            int(t.strip()) for t in str(raw).split(",")
            if t.strip().isdigit() and int(t.strip()) != 0
        ]
        gold[n + 1] = parents
    return gold


# Feed the GOLD tuples rather than the predicted ones, so this measures dependency
# generation in isolation instead of compounding tuple-extraction error.
for item_id in ("video_001", "video_002", "video_003"):
    text, query, gold_tuples, _ = gold_for(item_id)
    gold_block = "\n".join(f"{i + 1} | {t}" for i, t in enumerate(gold_tuples))
    p = fifa.make_prompt(
        examples=build_examples("dependency", exclude_ids={item_id}),
        test_input="\n".join([text, gold_block]),
        test_query=query,
        preamble=fifa._DEPENDENCY_PREAMBLE,
        task="dependency",
    )
    pred, dropped = parse_dependencies(
        fifa.parse_with_input_name(completion(p, 1024)).strip(),
        valid_ids=set(range(1, len(gold_tuples) + 1)),
    )
    gold = gold_dependencies(item_id)

    exact = sum(1 for k in gold if set(pred.get(k, [])) == set(gold[k]))
    inter = sum(len(set(pred.get(k, [])) & set(gold[k])) for k in gold)
    union = sum(len(set(pred.get(k, [])) | set(gold[k])) for k in gold)
    print(f"\n{item_id}: {exact}/{len(gold)} exact parent-set match, "
          f"edge Jaccard {inter / union if union else 1.0:.2f}, dropped={len(dropped)}")
    for k in sorted(gold):
        mark = " " if set(pred.get(k, [])) == set(gold[k]) else "x"
        print(f"  {mark} {k:>3}  gold={gold[k] or '-'}  pred={pred.get(k, []) or '-'}")


## Controlled Hallucination Injection

In [ ]:
import json

HALLUCINATION_TYPES = {
    "entity": "Replace one entity (a person, animal, or object) with a different one.",
    "attribute": "Change one attribute (colour, size, material, or state) of an entity.",
    "location": "Change the location or setting in which the scene takes place.",
    "action": "Change one action or verb to a different, incompatible action.",
    "relation": "Change a spatial relationship between two entities, or remove one participant.",
    "temporal": "Reverse the order of two events that occur in sequence.",
}

INJECT_PROMPT = """You are constructing a controlled evaluation set for a video \
description faithfulness metric.

Given a description, introduce EXACTLY ONE factual error of the requested type. The \
edit must be minimal: change as few words as possible and leave the rest of the \
description untouched.

Error type: {type_name}
Instruction: {type_instruction}

Description:
{description}

Respond with only a JSON object and no other text:
{{"hallucination_type": "{type_name}",
  "original_span": "<the exact substring of the description you are replacing, copied verbatim>",
  "replacement_span": "<the text that replaces it>",
  "rationale": "<one sentence stating what is now false>"}}

The value of "original_span" must appear verbatim in the description exactly once."""


def inject(description: str, htype: str, attempts: int = 4):
    """Return one minimally-perturbed variant, or None if every attempt failed.

    The model only proposes a span pair; the edit is applied here with str.replace so
    exactly one localised change is made. A model that rewrites the whole description
    would make any score drop unattributable to the injected error.
    """
    for _ in range(attempts):
        raw = completion(
            INJECT_PROMPT.format(
                type_name=htype,
                type_instruction=HALLUCINATION_TYPES[htype],
                description=description,
            ),
            max_tokens=512,
        )
        try:
            payload = json.loads(raw[raw.index("{"):raw.rindex("}") + 1])
            original = payload["original_span"]
            replacement = payload["replacement_span"]
        except (ValueError, KeyError):
            continue
        if not original.strip() or original == replacement:
            continue
        if description.count(original) != 1:
            continue
        return {
            "variant": htype,
            "original_span": original,
            "replacement_span": replacement,
            "rationale": payload.get("rationale", ""),
            "description": description.replace(original, replacement, 1),
        }
    return None


In [ ]:
def run_fifa_stages(description: str, query: str = None, max_tokens: int = 2048):
    """Tuple -> question -> dependency for one description, using the same prompts
    and parsers as the walkthrough cells above."""
    query = QUERY if query is None else query

    tuple_p = fifa.make_prompt(
        examples=fifa._TUPLE_EXAMPLES, test_input=description, test_query=query,
        preamble=fifa._TUPLE_PREAMBLE, task="tuple",
    )
    raw_t = fifa.parse_with_input_name(completion(tuple_p, max_tokens)).strip()
    facts, dropped_t = parse_tuples(raw_t)
    ids = {f["id"] for f in facts}

    question_p = fifa.make_prompt(
        examples=fifa._QUESTION_EXAMPLES,
        test_input="\n".join([description, raw_t]), test_query=query,
        preamble=fifa._QUESTION_PREAMBLE, task="question",
    )
    raw_q = fifa.parse_with_input_name(completion(question_p, max_tokens)).strip()
    questions, dropped_q = parse_questions(raw_q)

    dependency_p = fifa.make_prompt(
        examples=fifa._DEPENDENCY_EXAMPLES,
        test_input="\n".join([description, raw_t]), test_query=query,
        preamble=fifa._DEPENDENCY_PREAMBLE, task="dependency",
    )
    raw_d = fifa.parse_with_input_name(completion(dependency_p, max_tokens)).strip()
    deps, dropped_d = parse_dependencies(raw_d, valid_ids=ids)

    return {
        "description": description,
        "facts": facts,
        "questions": questions,
        "deps": deps,
        "raw": {"tuples": raw_t, "questions": raw_q, "dependencies": raw_d},
        "n_dropped": len(dropped_t) + len(dropped_q) + len(dropped_d),
    }


In [ ]:
variants = {}
for _htype in HALLUCINATION_TYPES:
    _v = inject(DESCRIPTION, _htype)
    if _v is None:
        print(f"{_htype:10s} FAILED after retries")
        continue
    variants[_htype] = _v
    print(f"{_htype:10s} {_v['original_span']!r}  ->  {_v['replacement_span']!r}")
    print(f"           {_v['rationale']}")

print(f"\n{len(variants)}/{len(HALLUCINATION_TYPES)} variants generated")


In [ ]:
records = {"original": {"variant": "original", "description": DESCRIPTION}}
records.update(variants)

pipelines = {}
for _name, _rec in records.items():
    _out = run_fifa_stages(_rec["description"])
    _out.update({k: _rec.get(k) for k in ("variant", "original_span", "replacement_span")})

    # Did the injected error survive into the extracted facts? If not, FIFA cannot
    # possibly detect it, and that is a finding about the extractor rather than the
    # verifier.
    if _rec.get("replacement_span"):
        _words = [w for w in re.findall(r"[a-z]+", _rec["replacement_span"].lower())
                  if len(w) > 3]
        _out["perturbation_in_facts"] = [
            f["id"] for f in _out["facts"]
            if any(w in f["text"].lower() for w in _words)
        ]
    else:
        _out["perturbation_in_facts"] = None

    pipelines[_name] = _out
    print(f"{_name:10s} facts={len(_out['facts']):>3} "
          f"questions={len(_out['questions']):>3} dropped={_out['n_dropped']:>2} "
          f"perturbation in facts: {_out['perturbation_in_facts']}")


# Verify Facts

In [ ]:
for _name in ("glm_struct", "glm_model", "glm_tokenizer"):
    globals().pop(_name, None)

cleanup_gpu()

if torch.cuda.is_available():
    print(f"allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB, "
          f"reserved: {torch.cuda.memory_reserved() / 1e9:.1f} GB")


In [ ]:
from transformers import Qwen3VLMoeForConditionalGeneration, AutoProcessor

VQA_MODEL = "Qwen/Qwen3-VL-30B-A3B-Instruct"

vqa_processor = AutoProcessor.from_pretrained(VQA_MODEL)
vqa_model = Qwen3VLMoeForConditionalGeneration.from_pretrained(
    VQA_MODEL, dtype="auto", device_map="auto"
)

# Same frame-sampling contract as the description stage: we hand over pre-sampled
# frames, so the processor must not resample, and it needs the real metadata or the
# per-frame timestamps written into the prompt will be wrong.
vqa_vp = vqa_processor.video_processor
vqa_vp.do_sample_frames = False
vqa_vp.video_metadata = [metadata]


def answer(question: str, video_frames=None, max_new_tokens: int = 128) -> str:
    video_frames = frames if video_frames is None else video_frames
    messages = [{
        "role": "user",
        "content": [
            {"type": "video", "video": video_frames},
            {"type": "text", "text": question},
        ],
    }]
    text = vqa_processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vqa_processor(
        text=text, videos=[video_frames], padding=True, return_tensors="pt"
    ).to(vqa_model.device)
    out = vqa_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return vqa_processor.tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


print(answer("Is there a bottle?"))


In [ ]:
_AFFIRM = {"yes", "yeah", "yep", "correct", "true", "affirmative"}
_DENY = {"no", "nope", "not", "false", "incorrect", "none", "never"}


def fifa_score(ans: str) -> float:
    """Verbatim reproduction of eval_video2text.py:63."""
    return float("yes" in ans.lower())


def strict_score(ans: str):
    """Return 1.0 / 0.0, or None when the answer is genuinely ambiguous.

    eval_video2text.py:63 does a bare substring test, so "yesterday" scores 1.0 and
    "There is no dog, yes I am sure" scores 1.0. Leading token wins here instead.
    """
    text = re.sub(r"[^a-z' ]+", " ", ans.lower()).strip()
    tokens = text.split()
    if not tokens:
        return None
    if tokens[0] in _AFFIRM:
        return 1.0
    if tokens[0] in _DENY:
        return 0.0
    if re.search(r"\bthere (is|are) no\b|\b(does|do|did) not\b|\bis not\b", text):
        return 0.0
    if re.search(r"\byes\b", text):
        return 1.0
    if re.search(r"\bno\b", text):
        return 0.0
    return None


def propagate_reference(scores, deps):
    """Single pass in dict order, reproducing eval_video2text.py:68-84.

    Order-dependent by construction: a parent zeroed after its child was visited
    does not zero that child. Kept so the reference number is reportable.
    """
    scores, validity = dict(scores), {}
    for did, parents in deps.items():
        if did not in scores:
            continue
        if any(scores.get(p) == 0 for p in parents):
            scores[did], validity[did] = 0.0, False
        else:
            validity[did] = True
    return scores, validity


def propagate_transitive(scores, deps):
    """Iterate to a fixpoint so zeroing propagates down the whole chain."""
    scores, validity = dict(scores), {d: True for d in deps if d in scores}
    changed = True
    while changed:
        changed = False
        for did, parents in deps.items():
            if did not in scores or scores[did] == 0:
                continue
            if any(scores.get(p) == 0 for p in parents):
                scores[did], validity[did] = 0.0, False
                changed = True
    return scores, validity


In [ ]:
scored = [r for r in stsdg if r["question"]]
assert scored, "no questions to score; re-run the question generation cell"

answers = {r["id"]: answer(r["question"]) for r in scored}

raw_fifa = {i: fifa_score(a) for i, a in answers.items()}
raw_strict = {i: strict_score(a) for i, a in answers.items()}

ambiguous = [i for i, s in raw_strict.items() if s is None]
raw_strict_filled = {i: (0.0 if s is None else s) for i, s in raw_strict.items()}

ref_scores, ref_validity = propagate_reference(raw_fifa, deps_a)
tr_scores, tr_validity = propagate_transitive(raw_strict_filled, deps_a)


def mean(d):
    return sum(d.values()) / len(d) if d else float("nan")


print(f"{'id':>3}  {'score':>5}  {'valid':>5}  question / answer")
for r in scored:
    i = r["id"]
    print(f"{i:>3}  {tr_scores[i]:>5.0f}  {str(tr_validity.get(i, True)):>5}  {r['question']}")
    print(f"       -> {answers[i][:90]}")

print()
print(f"ambiguous answers        : {len(ambiguous)} {ambiguous}")
print(f"fifa vs strict disagree  : "
      f"{sorted(i for i in raw_fifa if raw_fifa[i] != raw_strict_filled[i])}")
print()
print(f"no-dependency  (reference metric) : {mean(raw_fifa):.3f}")
print(f"single-pass    (reference metric) : {mean(ref_scores):.3f}")
print(f"transitive     (corrected)        : {mean(tr_scores):.3f}")
print(f"single-pass vs transitive differ  : "
      f"{sorted(i for i in ref_scores if ref_scores[i] != tr_scores[i])}")

faithfulness = {
    "n_facts": len(scored),
    "n_ambiguous": len(ambiguous),
    "average_score_without_dep": mean(raw_fifa),
    "average_score_reference": mean(ref_scores),
    "average_score_transitive": mean(tr_scores),
    "answers": answers,
    "scores": tr_scores,
    "validity": tr_validity,
}


In [ ]:
def score_record(rec, video_frames=None):
    scored = [f for f in rec["facts"] if f["id"] in rec["questions"]]
    if not scored:
        return None
    answers = {f["id"]: answer(rec["questions"][f["id"]], video_frames) for f in scored}

    strict_raw = {i: strict_score(a) for i, a in answers.items()}
    filled = {i: (0.0 if s is None else s) for i, s in strict_raw.items()}
    transitive, validity = propagate_transitive(filled, rec["deps"])
    reference, _ = propagate_reference(
        {i: fifa_score(a) for i, a in answers.items()}, rec["deps"]
    )
    return {
        "n_facts": len(scored),
        "n_ambiguous": sum(1 for s in strict_raw.values() if s is None),
        "average_score_without_dep": sum(filled.values()) / len(filled),
        "average_score_reference": sum(reference.values()) / len(reference),
        "average_score_transitive": sum(transitive.values()) / len(transitive),
        "answers": answers,
        "scores": transitive,
        "validity": validity,
    }


variant_results = {}
for _name, _rec in pipelines.items():
    variant_results[_name] = score_record(_rec)
    print(f"scored {_name}")

_base = variant_results["original"]
print()
print(f"{'variant':10s} {'facts':>5} {'score':>6} {'delta':>7} {'ref':>6} "
      f"{'refdelta':>8}  perturbation reached facts")
print(f"{'original':10s} {_base['n_facts']:>5} "
      f"{_base['average_score_transitive']:>6.3f} {'':>7} "
      f"{_base['average_score_reference']:>6.3f} {'':>8}")

_detected = 0
for _name, _r in variant_results.items():
    if _name == "original" or _r is None:
        continue
    _d = _r["average_score_transitive"] - _base["average_score_transitive"]
    _rd = _r["average_score_reference"] - _base["average_score_reference"]
    _reached = pipelines[_name]["perturbation_in_facts"]
    _detected += _d < 0
    print(f"{_name:10s} {_r['n_facts']:>5} {_r['average_score_transitive']:>6.3f} "
          f"{_d:>+7.3f} {_r['average_score_reference']:>6.3f} {_rd:>+8.3f}  "
          f"{_reached if _reached else 'NOT EXTRACTED'}")

_n = len(variant_results) - 1
print(f"\nFIFA scored {_detected}/{_n} corrupted descriptions below the original")
print("Variants whose injected error never reached the fact set are untestable by "
      "FIFA; count them separately from genuine misses.")
